# EDA Pollen & Météo — Analyse Complète 2021-2026
**Projet Antihistaminiques France — Jedha 2026**

## Sources
- **Pollen RNSA** (2021-2022) — mesures physiques par station
- **Pollen CAMS** (2023-2026) — modélisation spatiale Copernicus 0.1°
- **Météo Open-Meteo** (2021-2026) — ERA5 — 13 régions AASQA

## Plan
1. Chargement & aperçu
2. Evolution temporelle 2021-2026 — RNSA vs CAMS
3. Saisonnalité par mois
4. Heatmap zones géographiques
5. Carte France 13 régions
6. Météo par région
7. Corrélation météo / pollen
8. Random Forest Regressor — avec météo
9. Random Forest Regressor — sans météo (features temporelles uniquement)
10. Synthèse

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Chargement
df_pollen = pd.read_csv('/Users/nellyta/Jedha/data/silver/J0_silver_pollen_2021_2026.csv')
df_meteo  = pd.read_csv('/Users/nellyta/Jedha/data/silver/J0_silver_meteo_openmeteo_2021_2026.csv')

df_pollen['date'] = pd.to_datetime(df_pollen['date'])
df_meteo['time']  = pd.to_datetime(df_meteo['time'], format='mixed')
df_meteo['time']  = df_meteo['time'].dt.normalize()

# Agregation pollen
pollen_daily = df_pollen.groupby(['date','source']).agg(
    graminees=('graminees_conc','mean'),
    bouleau=('bouleau_conc','mean'),
    aulne=('aulne_conc','mean'),
    ambroisie=('ambroisie_conc','mean'),
    armoise=('armoise_conc','mean'),
    olivier=('olivier_conc','mean')
).reset_index()

pollen_daily['mois']             = pollen_daily['date'].dt.month
pollen_daily['annee']            = pollen_daily['date'].dt.year
pollen_daily['jour_annee']       = pollen_daily['date'].dt.dayofyear
pollen_daily['saison_allergies'] = pollen_daily['mois'].apply(lambda m: 1 if m in [4,5,6,7] else 0)
pollen_daily['source_encoded']   = (pollen_daily['source'] == 'CAMS').astype(int)

# Agregation meteo
meteo_daily = df_meteo.groupby('time').agg(
    temp_moy=('temperature_2m_mean','mean'),
    temp_max=('temperature_2m_max','mean'),
    temp_min=('temperature_2m_min','mean'),
    precip=('precipitation_sum','mean'),
    wind=('wind_speed_10m_max','mean')
).reset_index().rename(columns={'time':'date'})

merged = pollen_daily.merge(meteo_daily, on='date', how='left')

mois_noms  = {1:'Jan',2:'Fev',3:'Mar',4:'Avr',5:'Mai',6:'Jun',
              7:'Jul',8:'Aou',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
colors_src = {'RNSA':'gray','CAMS':'green'}
taxons_list = ['graminees','bouleau','aulne','ambroisie','armoise','olivier']

print(f'Pollen   : {pollen_daily.shape} | Sources : {pollen_daily.source.value_counts().to_dict()}')
print(f'Meteo    : {meteo_daily.shape}')
print(f'Fusionne : {merged.shape}')
print(f'Periode  : {merged.date.min().date()} -> {merged.date.max().date()}')

## 1. Evolution temporelle 2021-2026 — RNSA vs CAMS
> Pointillés = RNSA (2021-2022) / Trait plein = CAMS (2023-2026) / Zone grisée = période RNSA

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle('Evolution concentrations polliniques 2021-2026 (RNSA + CAMS)',
             fontsize=14, fontweight='bold')

taxons_axes = {
    'graminees':('Graminees',axes[0,0]),
    'bouleau':  ('Bouleau',  axes[0,1]),
    'aulne':    ('Aulne',    axes[1,0]),
    'ambroisie':('Ambroisie',axes[1,1]),
    'armoise':  ('Armoise',  axes[2,0]),
    'olivier':  ('Olivier',  axes[2,1]),
}

for col, (label, ax) in taxons_axes.items():
    for source in ['RNSA','CAMS']:
        df_s = pollen_daily[pollen_daily['source']==source]
        ax.plot(df_s['date'], df_s[col], color=colors_src[source],
                linewidth=0.8, linestyle='--' if source=='RNSA' else '-',
                label=source, alpha=0.8)
    ax.axvspan(pd.Timestamp('2021-01-01'), pd.Timestamp('2022-12-31'), alpha=0.05, color='gray')
    ax.set_title(label, fontweight='bold')
    ax.set_ylabel('Concentration (grains/m3)')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)
    max_val  = pollen_daily[col].max()
    max_date = pollen_daily.loc[pollen_daily[col].idxmax(),'date']
    ax.annotate(f'Max: {max_val:.1f}', xy=(max_date, max_val),
                xytext=(10,-20), textcoords='offset points', fontsize=7)

plt.tight_layout()
plt.savefig('/Users/nellyta/Jedha/notebooks/pollen_evolution_complet.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Saisonnalite par mois — RNSA vs CAMS

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Saisonnalite graminees par mois — RNSA vs CAMS', fontsize=13, fontweight='bold')

for idx, source in enumerate(['RNSA','CAMS']):
    df_s   = pollen_daily[pollen_daily['source']==source]
    saison = df_s.groupby('mois')['graminees'].mean()
    axes[idx].bar([mois_noms[m] for m in saison.index], saison.values,
                  color=colors_src[source], alpha=0.8)
    axes[idx].set_title(f'{source} — Graminees moyennes par mois', fontweight='bold')
    axes[idx].set_ylabel('Concentration (grains/m3)')
    axes[idx].grid(True, alpha=0.3, axis='y')
    for i, (m, v) in enumerate(saison.items()):
        axes[idx].text(i, v+0.5, f'{v:.1f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('/Users/nellyta/Jedha/notebooks/pollen_saisonnalite_complet.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Heatmap — Graminees par zone geographique et mois

In [ ]:
# Utiliser CAMS uniquement pour la heatmap spatiale
df_cams = df_pollen[df_pollen['source']=='CAMS'].copy()
df_cams['mois'] = df_cams['date'].dt.month
df_cams['zone'] = pd.cut(df_cams['latitude'],
    bins=[42,44,46,48,51],
    labels=['Sud (42-44)','Centre-Sud (44-46)','Centre-Nord (46-48)','Nord (48-51)'])

heatmap_data = df_cams.groupby(['zone','mois'])['graminees_conc'].mean().unstack()
heatmap_data.columns = [mois_noms[m] for m in heatmap_data.columns]

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Graminees — concentration par zone et mois (CAMS 2023-2026)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/Users/nellyta/Jedha/notebooks/heatmap_pollen_complet.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Carte France 13 regions AASQA (juin — CAMS)

In [ ]:
mapping_dept_region = {
    'Paris':'Ile-de-France','Hauts-de-Seine':'Ile-de-France',
    'Seine-Saint-Denis':'Ile-de-France',"Val-d'Oise":'Ile-de-France',
    'Yvelines':'Ile-de-France','Essonne':'Ile-de-France',
    'Seien-et-Marne':'Ile-de-France','Val-de-Marne':'Ile-de-France',
    'Nord':'Hauts-de-France','Pas-de-Calais':'Hauts-de-France',
    'Somme':'Hauts-de-France','Aisne':'Hauts-de-France','Oise':'Hauts-de-France',
    'Ardennes':'Grand Est','Meuse':'Grand Est','Meurthe-et-Moselle':'Grand Est',
    'Moselle':'Grand Est','Bas-Rhin':'Grand Est','Haute-Rhin':'Grand Est',
    'Vosges':'Grand Est','Marne':'Grand Est','Haute-Marne':'Grand Est','Aube':'Grand Est',
    'Seine-Maritime':'Normandie','Eure':'Normandie','Calvados':'Normandie',
    'Manche':'Normandie','Orne':'Normandie',
    'Finistere':'Bretagne',"Cotes-d'Armor":'Bretagne',
    'Morbihan':'Bretagne','Ille-et-Vilaine':'Bretagne',
    'Loire-Atlantique':'Pays-de-la-Loire','Vendee':'Pays-de-la-Loire',
    'Maine-et-Loire':'Pays-de-la-Loire','Sarthe':'Pays-de-la-Loire','Mayenne':'Pays-de-la-Loire',
    'Loiret':'Centre-Val-de-Loire','Loir-et-Cher':'Centre-Val-de-Loire',
    'Indre-et-Loire':'Centre-Val-de-Loire','Indre':'Centre-Val-de-Loire',
    'Cher':'Centre-Val-de-Loire','Eure-et-Loir':'Centre-Val-de-Loire',
    "Cote-d'Or":'Bourgogne-Franche-Comte','Nievre':'Bourgogne-Franche-Comte',
    'Saone-et-Loire':'Bourgogne-Franche-Comte','Yonne':'Bourgogne-Franche-Comte',
    'Doubs':'Bourgogne-Franche-Comte','Jura':'Bourgogne-Franche-Comte',
    'Haute-Saone':'Bourgogne-Franche-Comte','Territoire de Belfort':'Bourgogne-Franche-Comte',
    'Gironde':'Nouvelle-Aquitaine','Dordogne':'Nouvelle-Aquitaine',
    'Lot-et-Garonne':'Nouvelle-Aquitaine','Landes':'Nouvelle-Aquitaine',
    'Pyrenees-Atlantiques':'Nouvelle-Aquitaine','Charente':'Nouvelle-Aquitaine',
    'Charente-Maritime':'Nouvelle-Aquitaine','Deux-Sevres':'Nouvelle-Aquitaine',
    'Vienne':'Nouvelle-Aquitaine','Haute-Vienne':'Nouvelle-Aquitaine',
    'Correze':'Nouvelle-Aquitaine','Creuse':'Nouvelle-Aquitaine',
    'Haute-Garonne':'Occitanie','Hautes-Pyrenees':'Occitanie',
    'Ariege':'Occitanie','Pyrenees-Orientales':'Occitanie','Aude':'Occitanie',
    'Herault':'Occitanie','Gard':'Occitanie','Lozere':'Occitanie',
    'Aveyron':'Occitanie','Lot':'Occitanie','Tarn':'Occitanie',
    'Tarn-et-Garonne':'Occitanie','Gers':'Occitanie',
    'Rhone':'Auvergne-Rhone-Alpes','Ain':'Auvergne-Rhone-Alpes',
    'Loire':'Auvergne-Rhone-Alpes','Isere':'Auvergne-Rhone-Alpes',
    'Drome':'Auvergne-Rhone-Alpes','Ardeche':'Auvergne-Rhone-Alpes',
    'Savoie':'Auvergne-Rhone-Alpes','Haute-Savoie':'Auvergne-Rhone-Alpes',
    'Puy-de-Dome':'Auvergne-Rhone-Alpes','Allier':'Auvergne-Rhone-Alpes',
    'Haute-Loire':'Auvergne-Rhone-Alpes','Cantal':'Auvergne-Rhone-Alpes',
    'Bouches-du-Rhone':'PACA','Var':'PACA','Vaucluse':'PACA',
    'Alpes-Maritimes':'PACA','Alpes-de-Haute-Provence':'PACA','Hautes-Alpes':'PACA',
    'Haute-Corse':'Corse','Corse-du-Sud':'Corse',
}

regions = gpd.read_file('https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_1_states_provinces.zip')
regions_fr = regions[regions['admin']=='France'].copy()
regions_fr['name_clean'] = regions_fr['name'].str.normalize('NFKD').str.encode('ascii',errors='ignore').str.decode('ascii')
regions_fr['region'] = regions_fr['name_clean'].map(
    {k.encode('ascii','ignore').decode('ascii'):v for k,v in mapping_dept_region.items()})
regions_fr = regions_fr[regions_fr['region'].notna()]
regions_13  = regions_fr.dissolve(by='region').reset_index()

juin = df_cams[df_cams['date'].dt.month==6].copy()
gdf_juin = gpd.GeoDataFrame(juin, geometry=gpd.points_from_xy(juin['longitude'],juin['latitude']), crs='EPSG:4326')
joined = gpd.sjoin(gdf_juin, regions_13[['region','geometry']], how='left', predicate='within')
conc_region = joined.groupby('region')['graminees_conc'].mean().reset_index()
conc_region.columns = ['region','graminees_moy']
regions_13 = regions_13.merge(conc_region, on='region', how='left')

villes = {'Paris':(48.85,2.35),'Lyon':(45.75,4.85),'Marseille':(43.30,5.37),
          'Toulouse':(43.60,1.44),'Bordeaux':(44.84,-0.58),'Nantes':(47.22,-1.55),
          'Strasbourg':(48.57,7.75),'Lille':(50.63,3.07)}

fig, ax = plt.subplots(figsize=(10, 12))
regions_13.plot(column='graminees_moy', cmap='YlOrRd', linewidth=1.5,
                edgecolor='white', ax=ax, legend=True,
                missing_kwds={'color':'lightgray'},
                legend_kwds={'label':'Concentration (grains/m3)', 'shrink':0.6})
for _, row in regions_13.iterrows():
    centroid = row['geometry'].centroid
    ax.annotate(row['region'], xy=(centroid.x,centroid.y),
                ha='center', fontsize=6, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.1', facecolor='white', alpha=0.6))
for ville, (lat, lon) in villes.items():
    ax.plot(lon, lat, 'o', color='black', markersize=5, zorder=10)
    ax.annotate(ville, xy=(lon,lat), xytext=(5,5), textcoords='offset points',
                fontsize=8, bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
ax.set_title('Graminees en juin par region AASQA (CAMS 2023-2026)', fontsize=13, fontweight='bold')
ax.set_xlim(-5.5, 10.5)
ax.set_ylim(41.0, 51.5)
ax.grid(True, alpha=0.2, linestyle='--')
plt.tight_layout()
plt.savefig('/Users/nellyta/Jedha/notebooks/carte_graminees_complet.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Meteo par region 2021-2026

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Meteo par region 2021-2026 — Open-Meteo', fontsize=14, fontweight='bold')

for region in df_meteo['region'].unique():
    subset = df_meteo[df_meteo['region']==region]
    axes[0].plot(subset['time'], subset['temperature_2m_mean'], linewidth=0.6, alpha=0.6, label=region)
    axes[1].plot(subset['time'], subset['precipitation_sum'],   linewidth=0.6, alpha=0.6, label=region)

for ax, title, ylabel in zip(axes,
    ['Temperature moyenne quotidienne par region','Precipitations quotidiennes par region'],
    ['Temperature (C)','Precipitations (mm)']):
    ax.axvspan(pd.Timestamp('2021-01-01'), pd.Timestamp('2022-12-31'), alpha=0.05, color='gray')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend(bbox_to_anchor=(1.05,1), fontsize=7)

plt.tight_layout()
plt.savefig('/Users/nellyta/Jedha/notebooks/meteo_regions_complet.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Correlation meteo / graminees — RNSA vs CAMS

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Correlation Meteo vs Graminees — RNSA vs CAMS', fontsize=13, fontweight='bold')

for idx, source in enumerate(['RNSA','CAMS']):
    df_s = merged[merged['source']==source].dropna(subset=['temp_moy','precip'])
    corr_t = df_s['temp_moy'].corr(df_s['graminees'])
    corr_p = df_s['precip'].corr(df_s['graminees'])

    axes[idx,0].scatter(df_s['temp_moy'], df_s['graminees'],
                        alpha=0.3, color=colors_src[source], s=5)
    axes[idx,0].set_xlabel('Temperature (C)')
    axes[idx,0].set_ylabel('Graminees (grains/m3)')
    axes[idx,0].set_title(f'{source} — Temp vs Graminees')
    axes[idx,0].grid(True, alpha=0.3)
    axes[idx,0].annotate(f'r = {corr_t:.2f}', xy=(0.05,0.95), xycoords='axes fraction',
                         fontsize=12, fontweight='bold', color=colors_src[source])

    axes[idx,1].scatter(df_s['precip'], df_s['graminees'],
                        alpha=0.3, color='steelblue', s=5)
    axes[idx,1].set_xlabel('Precipitations (mm)')
    axes[idx,1].set_ylabel('Graminees (grains/m3)')
    axes[idx,1].set_title(f'{source} — Precip vs Graminees')
    axes[idx,1].grid(True, alpha=0.3)
    axes[idx,1].annotate(f'r = {corr_p:.2f}', xy=(0.05,0.95), xycoords='axes fraction',
                         fontsize=12, fontweight='bold', color='steelblue')

plt.tight_layout()
plt.savefig('/Users/nellyta/Jedha/notebooks/correlation_complet.png', dpi=150, bbox_inches='tight')
plt.show()

for source in ['RNSA','CAMS']:
    df_s = merged[merged['source']==source].dropna(subset=['temp_moy','precip'])
    print(f'{source} — Corr temp : {df_s["temp_moy"].corr(df_s["graminees"]):.3f} | Corr precip : {df_s["precip"].corr(df_s["graminees"]):.3f}')

## 7. Random Forest Regressor — AVEC meteo
> Features : temp_moy, temp_max, temp_min, precip, wind, mois, jour_annee, saison_allergies, source_encoded

In [ ]:
features_meteo = ['temp_moy','temp_max','temp_min','precip','wind',
                  'mois','jour_annee','saison_allergies','source_encoded']

def run_rf(df, features, label):
    df = df.copy().sort_values('date').reset_index(drop=True)
    df['gram_next'] = df['graminees'].shift(-1)
    df = df.dropna(subset=['gram_next']+features)
    if len(df) < 20:
        print(f'  {label} : pas assez de donnees')
        return None,None,None,None
    X = df[features]
    y = df['gram_next']
    X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
    rf = RandomForestRegressor(n_estimators=100,random_state=42)
    rf.fit(X_train,y_train)
    y_pred = rf.predict(X_test)
    r2   = r2_score(y_test,y_pred)
    rmse = np.sqrt(mean_squared_error(y_test,y_pred))
    print(f'  {label:<30} R2={r2:.3f} | RMSE={rmse:.3f} | n={len(df)}')
    return y_test,y_pred,r2,rmse

print('=== RF AVEC METEO ===')
resultats_meteo = {}
y_t,y_p,r2,rmse = run_rf(merged, features_meteo, 'Toutes annees 2021-2026')
if r2 is not None:
    resultats_meteo['toutes'] = {'y_test':y_t,'y_pred':y_p,'r2':r2,'rmse':rmse,'label':'Toutes annees 2021-2026','color':'purple'}

colors_an = {2021:'steelblue',2022:'coral',2023:'mediumseagreen',2024:'mediumpurple',2025:'orange',2026:'red'}
for annee in sorted(merged['annee'].unique()):
    df_an = merged[merged['annee']==annee]
    y_t,y_p,r2_,rmse_ = run_rf(df_an, features_meteo, f'Annee {annee}')
    if r2_ is not None:
        resultats_meteo[annee] = {'y_test':y_t,'y_pred':y_p,'r2':r2_,'rmse':rmse_,'label':f'Annee {annee}','color':colors_an.get(annee,'purple')}

In [ ]:
def plot_rf_bloc(res, features, suffix=''):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"RF Regressor — {res['label']}", fontsize=13, fontweight='bold')

    axes[0].scatter(res['y_test'], res['y_pred'], alpha=0.5, color=res['color'], s=20)
    max_val = max(float(res['y_test'].max()), float(res['y_pred'].max()))
    axes[0].plot([0,max_val],[0,max_val],'r--',linewidth=1)
    axes[0].set_xlabel('Reel (grains/m3)')
    axes[0].set_ylabel('Predit (grains/m3)')
    axes[0].set_title(f"R2={res['r2']:.3f} | RMSE={res['rmse']:.3f}")
    axes[0].grid(True, alpha=0.3)

    df_tmp = merged.copy().sort_values('date').reset_index(drop=True)
    df_tmp['gram_next'] = df_tmp['graminees'].shift(-1)
    df_tmp = df_tmp.dropna(subset=['gram_next']+features)
    rf_tmp = RandomForestRegressor(n_estimators=100,random_state=42)
    rf_tmp.fit(df_tmp[features], df_tmp['gram_next'])
    imp = pd.DataFrame({'feature':features,'importance':rf_tmp.feature_importances_}).sort_values('importance',ascending=True)
    axes[1].barh(imp['feature'], imp['importance'], color=res['color'])
    axes[1].set_title('Feature Importance')
    axes[1].grid(True, alpha=0.3, axis='x')

    plt.tight_layout()
    fname = res['label'].replace(' ','_').replace('-','_')
    plt.savefig(f"/Users/nellyta/Jedha/notebooks/rf_complet_{suffix}_{fname}.png", dpi=150, bbox_inches='tight')
    plt.show()

for key in resultats_meteo:
    plot_rf_bloc(resultats_meteo[key], features_meteo, suffix='meteo')

## 8. Random Forest Regressor — SANS meteo (demande prof)
> Features : mois, jour_annee, saison_allergies, source_encoded — uniquement temporelles

In [ ]:
features_sans_meteo = ['mois','jour_annee','saison_allergies','source_encoded']

print('=== RF SANS METEO ===')
resultats_sans = {}
y_t,y_p,r2,rmse = run_rf(merged, features_sans_meteo, 'Toutes annees 2021-2026')
if r2 is not None:
    resultats_sans['toutes'] = {'y_test':y_t,'y_pred':y_p,'r2':r2,'rmse':rmse,'label':'Toutes annees 2021-2026','color':'steelblue'}

for annee in sorted(merged['annee'].unique()):
    df_an = merged[merged['annee']==annee]
    y_t,y_p,r2_,rmse_ = run_rf(df_an, features_sans_meteo, f'Annee {annee}')
    if r2_ is not None:
        resultats_sans[annee] = {'y_test':y_t,'y_pred':y_p,'r2':r2_,'rmse':rmse_,'label':f'Annee {annee}','color':colors_an.get(annee,'steelblue')}

for key in resultats_sans:
    plot_rf_bloc(resultats_sans[key], features_sans_meteo, suffix='sans_meteo')

## 9. Synthese

In [ ]:
print('=' * 65)
print('SYNTHESE EDA POLLEN & METEO 2021-2026')
print('=' * 65)
print(f'  Periode   : {merged.date.min().date()} -> {merged.date.max().date()}')
print(f'  RNSA      : {(pollen_daily.source=="RNSA").sum()} jours')
print(f'  CAMS      : {(pollen_daily.source=="CAMS").sum()} jours')
print(f'  Regions   : {df_meteo["region"].nunique()}')
print()
print('PICS PAR TAXON')
for taxon in taxons_list:
    mois_pic = pollen_daily.groupby('mois')[taxon].mean().idxmax()
    val_max  = pollen_daily[taxon].max()
    print(f'  {taxon:<12} -> pic en {mois_noms[mois_pic]}, max = {val_max:.1f} grains/m3')
print()
print('CORRELATIONS METEO / GRAMINEES')
for source in ['RNSA','CAMS']:
    df_s = merged[merged['source']==source].dropna(subset=['temp_moy','precip'])
    print(f'  {source} — Temp : {df_s["temp_moy"].corr(df_s["graminees"]):.3f} | Precip : {df_s["precip"].corr(df_s["graminees"]):.3f}')
print()
print('RF AVEC METEO')
for k,v in resultats_meteo.items():
    print(f'  {v["label"]:<30} R2={v["r2"]:.3f} | RMSE={v["rmse"]:.3f}')
print()
print('RF SANS METEO')
for k,v in resultats_sans.items():
    print(f'  {v["label"]:<30} R2={v["r2"]:.3f} | RMSE={v["rmse"]:.3f}')